In [ ]:
## This document is about training a policy for multiple orientations
import torch
import sys
import time

from ttpi import TTPI 
from pivoting_wall import pwall_env
torch.set_default_dtype(torch.float64)
%load_ext autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device 

In [ ]:
# dim=6
state_min = torch.tensor([0, -torch.pi, -0.02, 0]).to(device) #[theta, theta_dot, py, target_ori]
state_max = torch.tensor([torch.pi/2, torch.pi, 0.02, torch.pi/2]).to(device)

n_state = 50
n_action = 50
mass = 0.16
g = 9.81
u_ps = 0.3

max_normal_force = 4* mass*g

action_min = torch.tensor([0, -u_ps*max_normal_force, 0]) #[f_1, f_2, py_dot]
action_max = torch.tensor([max_normal_force, u_ps*max_normal_force, 0.05])

In [ ]:
dt=0.001
dyn_system =  pwall_env(state_min, state_max, dt=dt, device=device)

# p_target = torch.tensor([dyn_system.l2-dyn_system.l1, dyn_system.l1-dyn_system.l2, 3*torch.pi/8, 0, 0,0,0, 0]).view(1,-1).to(device)


In [ ]:
domain_state = [torch.linspace(state_min[i],state_max[i],n_state).to(device) for i in range(len(state_max))]
domain_action = [torch.linspace(action_min[i],action_max[i],n_action).to(device) for i in range(len(action_max))]

In [ ]:
n_test = 100
dim_state = len(domain_state)
init_state = torch.empty((n_test,dim_state))
# for i in range(dim_state):
#     init_state[:,i] = state_min[i] + torch.rand(n_test).clip(0.25,0.75).to(device)*(state_max[i]-state_min[i])
init_state[:,-1] = state_min[-1] + torch.rand(n_test).to(device)*(state_max[-1]-state_min[-1])
state = init_state.to(device)

In [ ]:
def forward_model(state,action):
    next_state = dyn_system.forward_simulate(state,action,dt)
    return next_state


def reward(state,action):
    rewards = dyn_system.reward_state_action(state,action)
    return rewards

In [ ]:
# tol = torch.tensor([0.03, 0.03, 15/180*torch.pi]).to(device)[:3]

def callback(ttdp, state=state, file_name='fig',callback_count=0):
    print("Testing....")

    history = []
    T = 1000
    traj = state[:,:].clone()[:,None,:] 
    traj_actions = torch.empty(state.shape[0],T,3).to(device) #bsxTx3
    cum_reward = torch.tensor([0.]*state.shape[0]).to(device)
    dt_cum = 0
    for i in range(T):
        t0 = time.time()
        action = ttdp.policy(state)
        t1=time.time()
        dt_cum+=(t1-t0)
        r = ttdp.reward_normalized(state,action)
        cum_reward+=r
        state = forward_model(state,action)
        traj = torch.concat((traj,state[:,None,:]),dim=1)
        traj_actions[:,i,:]=action
    print("time taken by policy: ", dt_cum/T)


    plot_num = 1 # number of plotted tasks
    plt=dyn_system.animate_pivot(traj[-plot_num:].to('cpu').numpy(),traj_actions[-plot_num:].to('cpu').numpy(),
                      animation=True, step_skip=10, xmax=0.3,figsize=5, scale=10)

    plt.show()
    return r.mean().to("cpu"), cum_reward.mean().to("cpu")

In [ ]:
ttpi = TTPI(domain_state=domain_state,
                domain_action=domain_action,
                reward=reward,
                normalize_reward=True,
                forward_model=forward_model,
                gamma=0.99,
                rmax_v=100, rmax_a=100,
                nswp_v=5, nswp_a=5,
                kickrank_v=10, kickrank_a=10,
                max_batch_v=10**4,max_batch_a=10**5,
                eps_cross_v=1e-3,
                eps_cross_a=1e-3,
                eps_round_v=1e-3,
                eps_round_a=1e-3,
                n_samples=50,
                verbose=True,
                device=device) # action = 'deterministic_tt', 'stochastic_tt', 'random'

In [ ]:
resume= False
ttpi.train(resume=resume,
        callback=callback, callback_freq=10,
        verbose=False, file_name='pivoting')